# T7: セーフティカー影響分析 — 2026 R03 日本GP

**目的**: SC（Lap 22–27）前後のギャップ変動・ピット戦略・ポジション変動を定量化する

**入力データ**:
- `race_laps.csv` — 全ドライバー×全ラップ
- `race_control_messages.csv` — SC/VSCイベントタイムスタンプ
- `race_results.csv` — 最終レース結果

**出力**:
- `output/sc_impact_summary.csv`
- `output/sc_position_change.png`

In [ ]:
# ライブラリインポート
import matplotlib
matplotlib.use('Agg')  # GUIなし環境対応

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

print('ライブラリ読み込み完了')

In [ ]:
# ─── パス設定 ───────────────────────────────────────────────
BASE_DIR  = '/Volumes/lyssr_workspace/2026_1_4/Motorsports-Visualised'
EXPORT_DIR = os.path.join(BASE_DIR, 'data/2026_R03_Japan/export')
OUT_DIR   = os.path.join(BASE_DIR, 'notebooks/output')
os.makedirs(OUT_DIR, exist_ok=True)

# グラフスタイル
STYLE = {
    'bg_color':   '#1a1a2e',
    'text_color': '#ffffff',
    'grid_color': '#333355',
    'figsize':    (14, 8),
    'title_size': 16,
    'label_size': 12,
}

TYRE_COLORS = {
    'SOFT':         '#FF3333',
    'MEDIUM':       '#FFD700',
    'HARD':         '#FFFFFF',
    'INTERMEDIATE': '#39B54A',
    'WET':          '#0072CE',
    'UNKNOWN':      '#888888',
}

In [ ]:
# ─── 1. データ読み込み ───────────────────────────────────────
laps    = pd.read_csv(os.path.join(EXPORT_DIR, 'race_laps.csv'))
rcm     = pd.read_csv(os.path.join(EXPORT_DIR, 'race_control_messages.csv'))
results = pd.read_csv(os.path.join(EXPORT_DIR, 'race_results.csv'))

print(f'ラップ数: {len(laps)}行 / ドライバー数: {laps["Driver"].nunique()}')
print(f'レースコントロールメッセージ: {len(rcm)}行')
print(f'\nrace_control_messages.csv 先頭:')
rcm.head(10)

In [ ]:
# ─── 2. SC/VSC期間の特定 ────────────────────────────────────
sc_deployed = rcm[rcm['Message'].str.contains('SAFETY CAR DEPLOYED', na=False)]['Lap'].tolist()
sc_in_lap   = rcm[rcm['Message'].str.contains('SAFETY CAR IN THIS LAP', na=False)]['Lap'].tolist()
vsc_deploy  = rcm[rcm['Message'].str.contains('VSC DEPLOYED', na=False)]['Lap'].tolist()
vsc_ending  = rcm[rcm['Message'].str.contains('VSC ENDING', na=False)]['Lap'].tolist()

# SC期間ペア
sc_periods = []
for dep in sc_deployed:
    clear = next((c for c in sc_in_lap if c >= dep), None)
    sc_periods.append((dep, clear))

print(f'SC期間: {sc_periods}')
print(f'VSC期間: {[(d, next((e for e in vsc_ending if e >= d), None)) for d in vsc_deploy]}')

# メインSC期間
SC_START, SC_END = sc_periods[0] if sc_periods else (22, 27)

# 分析ウィンドウ
PRE_SC_START  = SC_START - 4
PRE_SC_END    = SC_START - 1
POST_SC_START = SC_END + 1
POST_SC_END   = SC_END + 4

print(f'\nSC期間: Lap {SC_START}–{SC_END}')
print(f'分析ウィンドウ — 前: Lap {PRE_SC_START}–{PRE_SC_END} / 後: Lap {POST_SC_START}–{POST_SC_END}')

In [ ]:
# ─── 3. 前処理 & SC中ピット特定 ─────────────────────────────
laps['LapNumber']   = laps['LapNumber'].astype(float).astype(int)
laps['TrackStatus'] = laps['TrackStatus'].astype(str).str.strip()

# クリーンラップ（60秒超、IsAccurate=True or SC中も含む）
laps_clean = laps[laps['LapTime_sec'] > 60].copy()

# SC期間中にピットインしたドライバー
sc_laps = laps[(laps['LapNumber'] >= SC_START) & (laps['LapNumber'] <= SC_END)]
pitted_during_sc = sc_laps[sc_laps['PitInTime_sec'].notna()]['Driver'].unique().tolist()

print(f'SC中ピット組 ({len(pitted_during_sc)}名): {pitted_during_sc}')

In [ ]:
# ─── 4. SC前後ポジション・コンパウンド・ペース取得 ──────────
def get_position_at_lap(df, lap_num):
    subset = df[df['LapNumber'] == lap_num][['Driver', 'Position']].dropna(subset=['Position'])
    subset = subset.copy()
    subset['Position'] = subset['Position'].astype(float).astype(int)
    return subset.set_index('Driver')['Position']

def get_compound_at_lap(df, lap_num):
    subset = df[df['LapNumber'] == lap_num][['Driver', 'Compound']]
    return subset.set_index('Driver')['Compound']

def get_avg_pace(df, lap_start, lap_end):
    """アウトラップ・インラップ除外、IsAccurate=Trueのラップ中央値"""
    subset = df[
        (df['LapNumber'] >= lap_start) &
        (df['LapNumber'] <= lap_end) &
        df['PitOutTime_sec'].isna() &
        df['PitInTime_sec'].isna() &
        (df['IsAccurate'] == True)
    ]
    return subset.groupby('Driver')['LapTime_sec'].median()

pos_before_sc  = get_position_at_lap(laps, PRE_SC_END)
pos_after_sc   = get_position_at_lap(laps, POST_SC_START)
comp_before_sc = get_compound_at_lap(laps, PRE_SC_END)
comp_after_sc  = get_compound_at_lap(laps, POST_SC_START)
pace_before    = get_avg_pace(laps_clean, PRE_SC_START, PRE_SC_END)
pace_after     = get_avg_pace(laps_clean, POST_SC_START, POST_SC_END)

print('データ取得完了')

In [ ]:
# ─── 5. サマリーDataFrame構築 ────────────────────────────────
all_drivers = results['Abbreviation'].tolist()
summary_rows = []

for drv in all_drivers:
    team = results.loc[results['Abbreviation'] == drv, 'TeamName'].values
    row = {
        'Driver':           drv,
        'Team':             team[0] if len(team) > 0 else 'Unknown',
        'PositionBeforeSC': int(pos_before_sc[drv]) if drv in pos_before_sc else None,
        'PositionAfterSC':  int(pos_after_sc[drv])  if drv in pos_after_sc  else None,
        'PittedDuringSC':   drv in pitted_during_sc,
        'CompoundBeforeSC': comp_before_sc.get(drv, 'UNKNOWN'),
        'CompoundAfterSC':  comp_after_sc.get(drv, 'UNKNOWN'),
        'PaceBeforeSC_sec': round(float(pace_before[drv]), 3) if drv in pace_before else None,
        'PaceAfterSC_sec':  round(float(pace_after[drv]),  3) if drv in pace_after  else None,
    }
    if row['PositionBeforeSC'] is not None and row['PositionAfterSC'] is not None:
        row['PositionChange'] = row['PositionBeforeSC'] - row['PositionAfterSC']
    else:
        row['PositionChange'] = None
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary['StrategicOutcome'] = summary['PositionChange'].apply(
    lambda x: 'Winner' if (x is not None and x > 0) else
              ('Loser' if (x is not None and x < 0) else 'No change')
)

output_cols = [
    'Driver', 'Team', 'PositionBeforeSC', 'PositionAfterSC',
    'PositionChange', 'PittedDuringSC', 'CompoundBeforeSC', 'CompoundAfterSC',
    'PaceBeforeSC_sec', 'PaceAfterSC_sec', 'StrategicOutcome'
]
summary = summary[output_cols]

csv_path = os.path.join(OUT_DIR, 'sc_impact_summary.csv')
summary.to_csv(csv_path, index=False)
print(f'CSV保存: {csv_path}')
summary

In [ ]:
# ─── 6. グラフ描画 ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor(STYLE['bg_color'])

def style_ax(ax, title):
    ax.set_facecolor(STYLE['bg_color'])
    ax.set_title(title, color=STYLE['text_color'], fontsize=14, fontweight='bold', pad=10)
    ax.tick_params(colors=STYLE['text_color'])
    ax.xaxis.label.set_color(STYLE['text_color'])
    ax.yaxis.label.set_color(STYLE['text_color'])
    for spine in ax.spines.values():
        spine.set_edgecolor(STYLE['grid_color'])
    ax.grid(color=STYLE['grid_color'], linestyle='--', alpha=0.5)

# パネル1: ラップタイム推移
ax1 = axes[0, 0]
style_ax(ax1, f'Lap Time Trend (Lap {PRE_SC_START}-{POST_SC_END})')
top10 = results.head(10)['Abbreviation'].tolist()
colors_p = plt.cm.tab10.colors
for i, drv in enumerate(top10):
    d = laps_clean[
        (laps_clean['Driver'] == drv) &
        (laps_clean['LapNumber'] >= PRE_SC_START) &
        (laps_clean['LapNumber'] <= POST_SC_END) &
        laps_clean['PitOutTime_sec'].isna() &
        laps_clean['PitInTime_sec'].isna()
    ].sort_values('LapNumber')
    if len(d) > 0:
        ax1.plot(d['LapNumber'], d['LapTime_sec'], marker='o', markersize=4,
                 linewidth=1.5, color=colors_p[i], label=drv, alpha=0.85)
ax1.axvspan(SC_START, SC_END, color='#ffff00', alpha=0.12, label=f'SC Lap {SC_START}-{SC_END}')
ax1.axvline(SC_START, color='#ffff00', linestyle='--', linewidth=1, alpha=0.7)
ax1.axvline(SC_END,   color='#ffff00', linestyle='--', linewidth=1, alpha=0.7)
ax1.set_xlabel('Lap', fontsize=STYLE['label_size'])
ax1.set_ylabel('Lap Time (s)', fontsize=STYLE['label_size'])
ax1.legend(fontsize=8, loc='upper right', framealpha=0.3,
           labelcolor=STYLE['text_color'], facecolor=STYLE['bg_color'])
ax1.invert_yaxis()

# パネル2: ポジション変動バー
ax2 = axes[0, 1]
style_ax(ax2, 'Position Change Before/After SC')
plot_df = summary.dropna(subset=['PositionChange', 'PositionBeforeSC']).copy()
plot_df = plot_df.sort_values('PositionBeforeSC')
bar_colors = [
    '#39B54A' if r['PittedDuringSC'] else
    '#3399FF' if r['PositionChange'] > 0 else
    '#FF6644' if r['PositionChange'] < 0 else '#888888'
    for _, r in plot_df.iterrows()
]
bars = ax2.bar(plot_df['Driver'], plot_df['PositionChange'],
               color=bar_colors, edgecolor='none', alpha=0.9)
for bar, (_, row) in zip(bars, plot_df.iterrows()):
    if row['PittedDuringSC']:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 'P', ha='center', va='bottom', color='#39B54A', fontsize=10, fontweight='bold')
ax2.axhline(0, color=STYLE['text_color'], linewidth=0.8)
ax2.set_xlabel('Driver', fontsize=STYLE['label_size'])
ax2.set_ylabel('Position Change (+up)', fontsize=STYLE['label_size'])
ax2.tick_params(axis='x', rotation=45)
ax2.legend(handles=[
    mpatches.Patch(color='#39B54A', label='Pitted during SC (P)'),
    mpatches.Patch(color='#3399FF', label='Gained'),
    mpatches.Patch(color='#FF6644', label='Lost'),
    mpatches.Patch(color='#888888', label='No change'),
], fontsize=9, framealpha=0.3, labelcolor=STYLE['text_color'], facecolor=STYLE['bg_color'])

# パネル3: ペース散布図
ax3 = axes[1, 0]
style_ax(ax3, 'Pace Comparison: Before vs After SC')
pace_df = summary.dropna(subset=['PaceBeforeSC_sec', 'PaceAfterSC_sec']).copy()
sc_colors = [
    '#39B54A' if r['PittedDuringSC'] else
    '#FF6644' if (r['PositionChange'] is not None and r['PositionChange'] < 0) else '#3399FF'
    for _, r in pace_df.iterrows()
]
ax3.scatter(pace_df['PaceBeforeSC_sec'], pace_df['PaceAfterSC_sec'],
            c=sc_colors, s=80, alpha=0.85, edgecolors='none', zorder=3)
for _, row in pace_df.iterrows():
    ax3.annotate(row['Driver'], (row['PaceBeforeSC_sec'], row['PaceAfterSC_sec']),
                 textcoords='offset points', xytext=(6, 2),
                 fontsize=7, color=STYLE['text_color'], alpha=0.85)
all_v = list(pace_df['PaceBeforeSC_sec']) + list(pace_df['PaceAfterSC_sec'])
mn, mx = min(all_v)-0.5, max(all_v)+0.5
ax3.plot([mn, mx], [mn, mx], color=STYLE['grid_color'], linestyle='--', linewidth=1, label='No change')
ax3.set_xlabel(f'Pre-SC Pace Median (Lap {PRE_SC_START}-{PRE_SC_END}) [s]', fontsize=STYLE['label_size'])
ax3.set_ylabel(f'Post-SC Pace Median (Lap {POST_SC_START}-{POST_SC_END}) [s]', fontsize=STYLE['label_size'])
ax3.legend(fontsize=9, framealpha=0.3, labelcolor=STYLE['text_color'], facecolor=STYLE['bg_color'])

# パネル4: タイヤ戦略
ax4 = axes[1, 1]
style_ax(ax4, 'Tyre Strategy — Before/After SC')
strat_df = summary.dropna(subset=['PositionBeforeSC']).sort_values('PositionBeforeSC').reset_index(drop=True)
n = len(strat_df)
for idx, (_, row) in enumerate(strat_df.iterrows()):
    cb = str(row['CompoundBeforeSC']).upper()
    ca = str(row['CompoundAfterSC']).upper()
    ax4.barh(idx, 0.45, left=0.0, color=TYRE_COLORS.get(cb, '#888888'), edgecolor='none', alpha=0.85, height=0.7)
    ax4.barh(idx, 0.45, left=0.55, color=TYRE_COLORS.get(ca, '#888888'), edgecolor='none', alpha=0.85, height=0.7)
    tc = '#000000' if cb == 'HARD' else STYLE['text_color']
    ax4.text(0.225, idx, cb[:3], ha='center', va='center', fontsize=7, color=tc, fontweight='bold')
    tc2 = '#000000' if ca == 'HARD' else STYLE['text_color']
    ax4.text(0.775, idx, ca[:3], ha='center', va='center', fontsize=7, color=tc2, fontweight='bold')
    if row['PittedDuringSC']:
        ax4.annotate('', xy=(0.55, idx), xytext=(0.45, idx),
                     arrowprops=dict(arrowstyle='->', color='#39B54A', lw=2))
ax4.set_yticks(range(n))
ax4.set_yticklabels([f"P{int(r['PositionBeforeSC'])} {r['Driver']}" for _, r in strat_df.iterrows()],
                    fontsize=9, color=STYLE['text_color'])
ax4.set_xlim(0, 1)
ax4.set_xticks([0.225, 0.775])
ax4.set_xticklabels(['Before SC', 'After SC'], fontsize=STYLE['label_size'])
ax4.grid(False)
ax4.legend(handles=[
    mpatches.Patch(color=TYRE_COLORS['SOFT'],   label='SOFT'),
    mpatches.Patch(color=TYRE_COLORS['MEDIUM'], label='MEDIUM'),
    mpatches.Patch(color=TYRE_COLORS['HARD'],   label='HARD'),
], fontsize=9, loc='lower right', framealpha=0.3,
   labelcolor=STYLE['text_color'], facecolor=STYLE['bg_color'])

fig.suptitle(
    f'2026 R03 Japan GP — Safety Car Impact Analysis\nSC Period: Lap {SC_START}–{SC_END}',
    color=STYLE['text_color'], fontsize=STYLE['title_size']+2, fontweight='bold', y=0.98
)
plt.tight_layout(rect=[0, 0, 1, 0.96])

png_path = os.path.join(OUT_DIR, 'sc_position_change.png')
fig.savefig(png_path, dpi=120, bbox_inches='tight', facecolor=STYLE['bg_color'])
plt.close()
print(f'グラフ保存: {png_path}')

In [ ]:
# ─── 7. コンソール出力サマリー ────────────────────────────
print('='*60)
print(f'=== SC Impact Summary (Lap {SC_START}-{SC_END}) ===')
print('='*60)
print(f'\nPitted during SC ({len(pitted_during_sc)}): {pitted_during_sc}')

winners = summary[summary['PositionChange'] > 0].sort_values('PositionChange', ascending=False)
losers  = summary[summary['PositionChange'] < 0].sort_values('PositionChange')

print(f'\nStrategic Winners ({len(winners)}):')
for _, r in winners.iterrows():
    pit = ' [PITTED]' if r['PittedDuringSC'] else ''
    print(f'  {r["Driver"]} ({r["Team"]}): P{r["PositionBeforeSC"]}->P{r["PositionAfterSC"]} (+{r["PositionChange"]}){pit}')

print(f'\nStrategic Losers ({len(losers)}):')
for _, r in losers.iterrows():
    pit = ' [PITTED]' if r['PittedDuringSC'] else ''
    print(f'  {r["Driver"]} ({r["Team"]}): P{r["PositionBeforeSC"]}->P{r["PositionAfterSC"]} ({r["PositionChange"]}){pit}')